In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.0
!pip install --no-deps trl==0.22.2

In [2]:
import os
import torch
import pandas as pd
from datasets import Dataset

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.9.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.9.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
model_name = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

max_seq_length = 16384  # or 4096 / 8192 depending on your GPU
dtype = torch.bfloat16

model, tokenizer = FastVisionModel.from_pretrained(
    model_name          = model_name,
    max_seq_length      = max_seq_length,
    load_in_4bit        = True,
    torch_dtype         = dtype,
    use_gradient_checkpointing = "unsloth",
)


==((====))==  Unsloth 2025.11.4: Fast Qwen3_Vl patching. Transformers: 4.57.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [4]:
model = FastVisionModel.get_peft_model(
    model,
    # In the video they leave vision layers frozen
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r           = 16,
    lora_alpha  = 16,
    lora_dropout= 0.0,
    bias        = "none",
    random_state= 3407,
    use_gradient_checkpointing = "unsloth"
)

try:
  model.print_trainable_parameters()
except Exception as e:
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  total = sum(p.numel() for p in model.parameters())
  print(f"Trainable: {trainable} | Total: {total} | {trainable/total*100:.2f}%")


trainable params: 43,646,976 || all params: 8,810,770,672 || trainable%: 0.4954


In [5]:
!unzip /content/TrarinImage.zip -d /content/Image

Archive:  /content/TrarinImage.zip
   creating: /content/Image/Image/
  inflating: /content/Image/Image/train0001.jpg  
  inflating: /content/Image/Image/train0002.jpg  
  inflating: /content/Image/Image/train0003.jpg  
  inflating: /content/Image/Image/train0004.jpg  
  inflating: /content/Image/Image/train0005.jpg  
  inflating: /content/Image/Image/train0006.jpg  
  inflating: /content/Image/Image/train0007.jpg  
  inflating: /content/Image/Image/train0008.jpg  
  inflating: /content/Image/Image/train0009.jpg  
  inflating: /content/Image/Image/train0010.jpg  
  inflating: /content/Image/Image/train0011.jpg  
  inflating: /content/Image/Image/train0012.jpg  
  inflating: /content/Image/Image/train0013.jpg  
  inflating: /content/Image/Image/train0014.jpg  
  inflating: /content/Image/Image/train0015.jpg  
  inflating: /content/Image/Image/train0016.jpg  
  inflating: /content/Image/Image/train0017.jpg  
  inflating: /content/Image/Image/train0018.jpg  
  inflating: /content/Image/Im

In [1]:
from datasets import Dataset
from PIL import Image
import pandas as pd
import os

CSV_PATH = "/content/Train.csv"
IMG_DIR  = "/content/Image/Image"

df = pd.read_csv(CSV_PATH)

TEXT_CONTENT = """You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:
Political: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.
NonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.
Output a single word without providing any reasoning or explanations"""

def make_example(row):
    img_path = os.path.join(IMG_DIR, row["Image_name"])
    if not os.path.exists(img_path):
        return None

    # Load + normalize image once
    try:
        img = Image.open(img_path)
        if img.mode != "RGB":
            img = img.convert("RGB")
        img = img.resize((512, 512))
    except:
        return None

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": TEXT_CONTENT},
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": row["Label"]},
            ],
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # target is included
    )

    return {"prompt": prompt, "image": img}

examples = []
total = 500
count = 0
for _, r in df.iterrows():
    ex = make_example(r)
    if ex is not None:
        examples.append(ex)
        count += 1
    if count == total:
      break

train_dataset = Dataset.from_list(examples)
print("Dataset size:", len(train_dataset))
print(train_dataset[0]["prompt"][:300])
print(type(train_dataset[0]["image"]))


NameError: name 'tokenizer' is not defined

In [ ]:
def preprocess(example):
    out = tokenizer(
        text   = example["prompt"],
        images = example["image"],
        return_tensors = None,
        padding = False,
    )
    # Ensure 1D lists, not list-of-lists
    if isinstance(out["input_ids"][0], list):
        out["input_ids"]      = out["input_ids"][0]
        out["attention_mask"] = out["attention_mask"][0]
    return out

tokenized_dataset = train_dataset.map(
    preprocess,
    remove_columns = train_dataset.column_names,  # drop prompt/image
)

print(tokenized_dataset[0].keys())
print(len(tokenized_dataset[0]["input_ids"]))


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [22]:
FastVisionModel.for_training(model)  # enable training

training_args = SFTConfig(
    output_dir                 = "qwen3_vl_meme_sft",
    logging_steps              = 10,
    save_steps                 = 0,
    learning_rate              = 5e-6,
    weight_decay               = 0.0,
    per_device_train_batch_size= 1,   # <- critical
    gradient_accumulation_steps= 8,   # effective batch 8
    num_train_epochs           = 1.0,
    max_steps                  = -1,
    dataset_text_field         = "",
    dataset_kwargs             = {"skip_prepare_dataset": True},
    seed                       = 3407,
    report_to                  = "none",
    remove_unused_columns      = False,
    fp16                       = not torch.cuda.is_bf16_supported(),
    bf16                       = torch.cuda.is_bf16_supported(),
)



In [ ]:
def has_one_image(example):
    msgs = example["messages"]
    for m in msgs:
        if m["role"] == "user":
            num_imgs = sum(1 for c in m["content"] if c.get("type") == "image")
            return num_imgs == 1
    return False

converted_dataset = converted_dataset.filter(has_one_image)
print("Size after image check:", len(converted_dataset))
for i in range(len(converted_dataset)):
   print(converted_dataset[i]["messages"])


In [21]:
def valid_example(example):
    msgs = example["messages"]

    # Must have at least one user and one assistant
    if len(msgs) < 2:
        return False
    user = msgs[0]
    assistant = msgs[1]

    if user.get("role") != "user" or assistant.get("role") != "assistant":
        return False

    # User: exactly 1 image, at least 1 text
    img_count = 0
    txt_count = 0
    for c in user.get("content", []):
        if c.get("type") == "image" and isinstance(c.get("image"), str):
            img_count += 1
        if c.get("type") == "text" and isinstance(c.get("text"), str):
            txt_count += 1
    if img_count != 1 or txt_count == 0:
        return False

    # Assistant: exactly 1 text
    a_txt = [c for c in assistant.get("content", []) if c.get("type") == "text" and isinstance(c.get("text"), str)]
    if len(a_txt) != 1:
        return False

    return True

converted_dataset = converted_dataset.filter(valid_example)
print("Size after strict filter:", len(converted_dataset))
print(converted_dataset[0]["messages"])


Filter:   0%|          | 0/2860 [00:00<?, ? examples/s]

Size after strict filter: 2860
[{'content': [{'image': '/content/Image/Image/train0001.jpg', 'text': None, 'type': 'image'}, {'image': None, 'text': 'You are an expert meme classifier. Classify the provided meme (which may be in Bengali, English or both) into the following category:\nPolitical: A meme depicting a political figure, making fun of a political figure, making fun of a political ideology or political party or mentioning a political figure.\nNonPolitical: A meme that is non political, contains humor or sarcasm regarding any non political issues or memes about relatable things or day to day life.\nOutput a single word without providing any reasoning or explanations', 'type': 'text'}], 'role': 'user'}, {'content': [{'image': None, 'text': 'Political', 'type': 'text'}], 'role': 'assistant'}]


In [23]:
def formatting_func(examples):
    # SFTTrainer calls this to build prompts from the dataset
    return examples["messages"]

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,                            # vision tokenizer handles text+images
    data_collator    = UnslothVisionDataCollator(model, tokenizer),  # must for Qwen3‑VL
    args             = training_args,
    train_dataset    = converted_dataset,
    formatting_func  = formatting_func,
)

train_results = trainer.train()
print(train_results)


Unsloth: Model does not have a default image size - using 512


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,860 | Num Epochs = 1 | Total steps = 358
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,810,770,672 (0.50% trained)


IndexError: index 1 is out of bounds for dimension 0 with size 1